In [6]:
# ==========================================================
# CELL 0: MOUNT GOOGLE DRIVE + CÀI ĐẶT OLLAMA + TẢI LLAMA 3.2 3B
# (Mount Drive để đọc file "Gen dataset.json" của bạn trên đó.
#  Ollama thì không cần Drive, chạy thẳng trên Colab.)
# ==========================================================

# 0. Mount Google Drive (file Gen dataset.json của bạn đang nằm trong
#    MyDrive/Project DAP/Gen dataset.json)
from google.colab import drive
drive.mount('/content/drive')

# 1. Cài công cụ giải nén zstd (Ollama cần để cài đặt đúng)
!apt-get install -y zstd

# 2. Cài Ollama vào máy ảo Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Chạy Ollama server dưới nền (background), lắng nghe ở localhost:11434
import subprocess
import time
subprocess.Popen(["ollama", "serve"])

# Chờ vài giây để service khởi động xong
time.sleep(5)

# 4. Tải model llama3.2:3b về Colab (đây là server Colab riêng, không liên quan
#    tới Ollama trên máy Windows của bạn — nó sẽ tự pull lại model trên cloud)
!ollama pull llama3.2:3b

# 5. "Làm nóng" model: gọi thử 1 lần để Ollama load hẳn model vào RAM/VRAM ngay bây giờ.
#    Nếu không làm bước này, request ĐẦU TIÊN ở Cell 1 sẽ phải tự load model
#    (mất khoảng 30s-1 phút) và dễ bị timeout. Lần gọi này có thể mất chút thời gian,
#    đó là bình thường — cứ để nó chạy xong.
import requests
print("Đang load model vào memory (làm nóng), có thể mất 30s-1 phút...")
try:
    warmup = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3.2:3b", "prompt": "Xin chào", "stream": False},
        timeout=180  # cho phép chờ tới 3 phút cho lần load đầu tiên
    )
    warmup.raise_for_status()
    print("✅ Model đã load xong vào memory, sẵn sàng trả lời nhanh ở các lần gọi sau.")
except Exception as e:
    print(f"⚠️ Làm nóng model thất bại: {e}")
    print("   Không sao, Cell 1 vẫn có thể tự thử lại, nhưng nếu lỗi lặp lại,")
    print("   chạy lại cell này (Cell 0) một lần nữa.")

print("\n✅ Ollama đã sẵn sàng, model llama3.2:3b đã tải và load xong trên Colab.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Đang load model vào memory (làm nóng), có thể mất 30s-1 phút...
✅ Model đã load xong vào memory, sẵn sàng trả lời nhanh ở các lần gọi sau.

✅ Ollama đã sẵn sàng, model llama3.2:3b đã tải và load xong trê

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ==========================================================
# CELL NÂNG CẤP: K-NN THRESHOLD + TÍCH HỢP LLM (LLAMA 3.2 / QWEN)
# ==========================================================

# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT (Bổ sung thư viện chạy LLM)
!pip install sentence-transformers scikit-learn pandas numpy tqdm ipywidgets requests -q

import json
import torch
import random
import requests  # Dùng để gọi Ollama API (thay cho transformers pipeline)
import numpy as np
import pandas as pd
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ==========================================================
# CẤU HÌNH THAM SỐ (Thay đổi ở đây để tối ưu)
# ==========================================================
K_NEIGHBORS = 3       # Số lượng câu tham gia biểu quyết K-NN
THRESHOLD = 0.40      # Ngưỡng độ tương đồng tối thiểu (Leader yêu cầu đặt threshold)

# MÔ HÌNH LLM: Llama 3.2 3B, chạy qua OLLAMA (đã cài ở Cell 0)
# Không cần token Hugging Face, không cần transformers/bitsandbytes cho phần này.
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL_NAME = "llama3.2:3b"   # phải khớp đúng tên model đã `ollama pull` ở Cell 0


# ==========================================================
# 2. ĐỌC VÀ CHIA DATASET (Giữ nguyên logic phân tầng của Linh)
# ==========================================================
print("--- 1. Đang tải và phân tách dữ liệu ---")
with open('/content/drive/MyDrive/Project DAP/Gen_dataset_v5.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

records = []
for intent in data['intents']:
    tag = intent['tag']
    for pattern in intent['patterns']:
        records.append({'pattern': pattern, 'intent': tag})

df = pd.DataFrame(records)
X = df['pattern']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)


# ==========================================================
# 3. VECTOR HÓA DỮ LIỆU (MÔ HÌNH EMBEDDING BGE-M3)
# ==========================================================
print("\n--- 2. Tải mô hình Embedding (BGE-M3) ---")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print("⚠️ Không phát hiện GPU — embedding sẽ chạy trên CPU (chậm hơn nhiều).")
    print("   Nếu muốn dùng GPU: Runtime > Change runtime type > GPU (T4) > Save > Restart session.")
else:
    print(f"✅ Đang dùng GPU: {torch.cuda.get_device_name(0)}")
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

print("\n--- 3. Đang chuyển đổi dữ liệu chữ sang Vector ---")
X_train_list = X_train.tolist()
X_test_list = X_test.tolist()
y_train_list = y_train.tolist()

X_train_vectors = embed_model.encode(X_train_list, show_progress_bar=True)
X_test_vectors = embed_model.encode(X_test_list, show_progress_bar=True)


# ==========================================================
# 4. KIỂM TRA KẾT NỐI TỚI OLLAMA (LLM chạy qua API, không load vào VRAM PyTorch)
# ==========================================================
print(f"\n--- 4. Kiểm tra Ollama đang chạy model ({OLLAMA_MODEL_NAME}) ---")
try:
    test_payload = {"model": OLLAMA_MODEL_NAME, "prompt": "Xin chào", "stream": False}
    test_response = requests.post(OLLAMA_URL, json=test_payload, timeout=120)
    test_response.raise_for_status()
    print("-> Kết nối Ollama thành công! LLM đã sẵn sàng nhận yêu cầu.")
except Exception as e:
    print(f"❌ Lỗi kết nối Ollama: {e}")
    print("   Kiểm tra lại: đã chạy Cell 0 (cài Ollama + ollama serve + ollama pull) chưa?")


# ==========================================================
# 5. CHẠY THỬ NGHIỆM TẬP TEST (Cập nhật THRESHOLD lọc nhiễu)
# ==========================================================
print(f"\n--- 5. Chạy RAG + K-NN (Threshold={THRESHOLD}) trên tập Test ---")
similarity_matrix = cosine_similarity(X_test_vectors, X_train_vectors)

y_pred = []
for i in range(len(X_test_list)):
    max_score = np.max(similarity_matrix[i])

    # NẾU ĐỘ TƯƠNG ĐỒNG THẤP HƠN NGƯỠNG THRESHOLD -> LOẠI THẲNG THÀNH "UNKNOWN"
    if max_score < THRESHOLD:
        y_pred.append("unknown_fallback")
    else:
        # Nếu vượt qua ngưỡng, lấy Top K câu để biểu quyết K-NN như bình thường
        top_k_indices = np.argsort(similarity_matrix[i])[-K_NEIGHBORS:]
        top_k_intents = [y_train_list[idx] for idx in top_k_indices]
        most_common_intent = Counter(top_k_intents).most_common(1)[0][0]
        y_pred.append(most_common_intent)

# Điều chỉnh nhãn tập Test để tính ma trận lỗi chuẩn xác (áp dụng nhãn unknown cho các câu bị lọc)
y_test_adjusted = [y_test.tolist()[i] if y_pred[i] != "unknown_fallback" else "unknown_fallback" for i in range(len(y_test))]

accuracy = accuracy_score(y_test_adjusted, y_pred)
f1 = f1_score(y_test_adjusted, y_pred, average='weighted')

print("\n" + "="*50)
print("          KẾT QUẢ ĐÁNH GIÁ SAU KHI CÓ THRESHOLD    ")
print("="*50)
print(f" - Accuracy (Độ chính xác tổng quan): {accuracy * 100:.2f}%")
# Lưu ý: Điểm số có thể giảm nhẹ vì ta đã "loại thẳng" các câu nghi vấn dưới ngưỡng, giúp hệ thống an toàn hơn.
print(f" - F1-Score (Trọng số tương quan):     {f1 * 100:.2f}%")
print("="*50)


# ==========================================================
# 6. GIAO DIỆN CHAT TRỰC TIẾP TÍCH HỢP LLM (RAG THỰC THỤ)
# ==========================================================
print("\n--- 6. GIAO DIỆN CHAT THỰC TẾ VỚI LLM ---")

intent_to_responses = {intent['tag']: intent.get('responses', []) for intent in data['intents']}

input_text = widgets.Text(placeholder='Hãy thử đổi từ, dùng câu đồng nghĩa...', description='Bạn:')
button = widgets.Button(description='Gửi câu chat')
output = widgets.Output()

# Tạo khung chứa chat lịch sử
output.layout.max_height = '400px'
output.layout.overflow = 'auto'
output.layout.border = '1px solid #ccc'
output.layout.padding = '10px'

with output:
    print("Bot RAG + LLM đã sẵn sàng. Hãy test độ nhạy ngữ nghĩa câu!")
    print("-" * 60)

def generate_llm_response(user_query, retrieved_context):
    """ Hàm gửi Prompt cho Llama 3.2 3B qua Ollama API, lấy câu trả lời dựa trên dữ liệu RAG """
    prompt = f'''You are a helpful and empathetic assistant. Answer the user's question based ONLY on the provided context. Keep it natural and concise (under 3 sentences).

Context: {retrieved_context}

Question: {user_query}

Answer:'''

    payload = {
        "model": OLLAMA_MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.7}
    }

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["response"].strip()
    except Exception as e:
        return f"(Lỗi gọi LLM qua Ollama: {e})"

def on_submit(b):
    user_input = input_text.value.strip()
    if not user_input:
        return
    with output:
        print(f"Bạn: {user_input}")

        # 1. Đo độ tương đồng ngữ nghĩa
        input_vector = embed_model.encode([user_input], device=device)
        sim_scores = cosine_similarity(input_vector, X_train_vectors)[0]
        highest_score = np.max(sim_scores)

        # 2. KIỂM TRA NGƯỠNG THRESHOLD THEO YÊU CẦU CỦA LEADER
        if highest_score < THRESHOLD:
            print(f"Bot: Xin lỗi, câu hỏi này nằm ngoài phạm vi hiểu biết hiện tại của hệ thống.")
            print(f"   \033[91m[Hệ thống từ chối - Độ tương đồng cao nhất: {highest_score:.2f} < Ngưỡng: {THRESHOLD}]\033[0m")
            print("-" * 60)
        else:
            # Nếu vượt ngưỡng, tiến hành dùng K-NN lấy dữ liệu
            top_k_indices = np.argsort(sim_scores)[-K_NEIGHBORS:]
            top_k_intents = [y_train_list[idx] for idx in top_k_indices]
            pred_intent = Counter(top_k_intents).most_common(1)[0][0]

            best_match_idx = top_k_indices[-1]
            matched_pattern = X_train_list[best_match_idx]

            # Lấy danh sách câu trả lời mẫu làm Ngữ cảnh (Context) cho LLM học
            context_responses = intent_to_responses.get(pred_intent, [""])
            context_text = " ".join(context_responses[:2]) # Lấy tối đa 2 câu trả lời mẫu làm nền tảng kiến thức

            # 3. ĐƯA VÀO LLM ĐỂ SINH CÂU TRẢ LỜI NGỮ NGHĨA TỰ NHIÊN
            print("Bot đang suy nghĩ...")
            bot_llm_response = generate_llm_response(user_input, context_text)

            # Xóa dòng chữ "Bot đang suy nghĩ..." bằng cách cập nhật giao diện
            clear_output(wait=True)
            print(f"Bạn: {user_input}")
            print(f"Bot (LLM sinh câu): {bot_llm_response}")
            print(f"   \033[92m[Debug - K-NN gán Tag: {pred_intent} | Khớp nhất: '{matched_pattern}' | Score: {highest_score:.2f}]\033[0m")
            print("-" * 60)

    input_text.value = ""

button.on_click(on_submit)
input_text.on_submit(on_submit)

display(widgets.VBox([output, widgets.HBox([input_text, button])]))

--- 1. Đang tải và phân tách dữ liệu ---

--- 2. Tải mô hình Embedding (BGE-M3) ---
✅ Đang dùng GPU: Tesla T4


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


--- 3. Đang chuyển đổi dữ liệu chữ sang Vector ---


Batches:   0%|          | 0/138 [00:00<?, ?it/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]


--- 4. Kiểm tra Ollama đang chạy model (llama3.2:3b) ---
-> Kết nối Ollama thành công! LLM đã sẵn sàng nhận yêu cầu.

--- 5. Chạy RAG + K-NN (Threshold=0.4) trên tập Test ---

          KẾT QUẢ ĐÁNH GIÁ SAU KHI CÓ THRESHOLD    
 - Accuracy (Độ chính xác tổng quan): 89.27%
 - F1-Score (Trọng số tương quan):     89.05%

--- 6. GIAO DIỆN CHAT THỰC TẾ VỚI LLM ---


In [ ]:
# ==========================================================
# 5B. TỐI ƯU K_NEIGHBORS BẰNG GRIDSEARCH + STRATIFIED K-FOLD
# ==========================================================
# Vì sao GridSearch (không dùng Optuna): K_NEIGHBORS là tham số RỜI RẠC,
# không gian tìm kiếm nhỏ (vài chục giá trị nguyên) -> GridSearch quét hết
# toàn bộ, cho kết quả chính xác tuyệt đối, không cần thuật toán tìm kiếm
# thông minh (Bayesian optimization) như Optuna vốn sinh ra cho không gian
# tham số lớn/liên tục/nhiều chiều.
#
# Vì sao cần Stratified K-Fold ở đây: 1 lần train_test_split chỉ cho 1 con
# số Accuracy/F1 duy nhất, có thể "ăn may" hoặc "không may" tùy cách chia
# (đã thấy dao động 88.5%-89.5% giữa các random_state khác nhau). K-Fold
# đánh giá mỗi giá trị K trên NHIỀU cách chia khác nhau, lấy trung bình
# -> kết quả đáng tin cậy hơn để chọn K tốt nhất.
#
# Lưu ý quan trọng: vẫn cần random_state cố định ở đây (cho StratifiedKFold)
# để việc chia fold lặp lại được giữa các lần chạy khác nhau -> so sánh các
# giá trị K một cách công bằng (cùng fold, chỉ đổi K).

from sklearn.model_selection import StratifiedKFold

K_CANDIDATES = list(range(1, 16))   # Quét K từ 1 đến 15
CV_FOLDS = 5
CV_RANDOM_STATE = 123  # Cố định để các fold giống nhau giữa các lần chạy, so sánh công bằng

print(f"--- Đang tối ưu K_NEIGHBORS bằng GridSearch + Stratified {CV_FOLDS}-Fold CV ---")
print(f"Các giá trị K sẽ thử: {K_CANDIDATES}\n")

# Mã hóa toàn bộ dataset 1 lần duy nhất (tránh encode lại ở mỗi fold, tốn thời gian)
print("Đang encode toàn bộ dataset (chỉ 1 lần)...")
X_all_list = X.tolist()
y_all_list = y.tolist()
X_all_vectors = embed_model.encode(X_all_list, show_progress_bar=True)
print(f"Đã encode xong {len(X_all_list)} câu.\n")

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_RANDOM_STATE)

def predict_knn(sim_matrix, y_train_fold, k, threshold=THRESHOLD):
    """Dự đoán nhãn bằng K-NN + threshold, giống đúng logic ở Cell chính."""
    preds = []
    for i in range(sim_matrix.shape[0]):
        max_score = np.max(sim_matrix[i])
        if max_score < threshold:
            preds.append("unknown_fallback")
        else:
            top_k_idx = np.argsort(sim_matrix[i])[-k:]
            top_k_labels = [y_train_fold[idx] for idx in top_k_idx]
            preds.append(Counter(top_k_labels).most_common(1)[0][0])
    return preds

# Lưu kết quả: mỗi K -> list điểm accuracy/f1 qua từng fold
results = {k: {"accuracy": [], "f1": []} for k in K_CANDIDATES}

fold_num = 0
for train_idx, val_idx in skf.split(X_all_list, y_all_list):
    fold_num += 1
    print(f"Đang xử lý Fold {fold_num}/{CV_FOLDS}...")

    X_train_fold_vec = X_all_vectors[train_idx]
    X_val_fold_vec = X_all_vectors[val_idx]
    y_train_fold = [y_all_list[i] for i in train_idx]
    y_val_fold = [y_all_list[i] for i in val_idx]

    # Tính similarity 1 lần cho fold này, dùng lại cho mọi giá trị K
    sim_matrix = cosine_similarity(X_val_fold_vec, X_train_fold_vec)

    for k in K_CANDIDATES:
        y_pred_fold = predict_knn(sim_matrix, y_train_fold, k)
        y_true_adjusted = [
            y_val_fold[i] if y_pred_fold[i] != "unknown_fallback" else "unknown_fallback"
            for i in range(len(y_val_fold))
        ]
        acc = accuracy_score(y_true_adjusted, y_pred_fold)
        f1 = f1_score(y_true_adjusted, y_pred_fold, average='weighted')
        results[k]["accuracy"].append(acc)
        results[k]["f1"].append(f1)

# ==========================================================
# TỔNG HỢP KẾT QUẢ: mean ± std qua 5 fold cho mỗi K
# ==========================================================
print("\n" + "="*70)
print(f"{'K':<5}{'Accuracy (mean ± std)':<28}{'F1-Score (mean ± std)':<28}")
print("="*70)

summary = []
for k in K_CANDIDATES:
    acc_mean = np.mean(results[k]["accuracy"]) * 100
    acc_std = np.std(results[k]["accuracy"]) * 100
    f1_mean = np.mean(results[k]["f1"]) * 100
    f1_std = np.std(results[k]["f1"]) * 100
    summary.append((k, acc_mean, acc_std, f1_mean, f1_std))
    print(f"{k:<5}{acc_mean:.2f}% ± {acc_std:.2f}%{'':<10}{f1_mean:.2f}% ± {f1_std:.2f}%")

# Chọn K tốt nhất theo F1-score trung bình (ưu tiên F1 vì dataset đa lớp,
# Accuracy có thể bị thiên vị nếu các lớp không hoàn toàn đều nhau)
best = max(summary, key=lambda x: x[3])
print("="*70)
print(f"\n✅ K_NEIGHBORS tốt nhất theo F1-score: K = {best[0]} "
      f"(F1 = {best[3]:.2f}% ± {best[4]:.2f}%, Accuracy = {best[1]:.2f}% ± {best[2]:.2f}%)")
print(f"\nGiá trị hiện tại trong Cell chính: K_NEIGHBORS = {K_NEIGHBORS}")
print(f"Nếu muốn áp dụng K tối ưu, sửa lại K_NEIGHBORS = {best[0]} ở Cell config rồi chạy lại từ đầu.")


In [9]:
# ==========================================================
# 5B. PHÂN TÍCH LỖI CHI TIẾT (Confusion Matrix + Top nhầm lẫn)
# Chạy SAU Cell chính (Cell 1), dùng lại y_test, y_pred đã có sẵn.
# ==========================================================
import pandas as pd
from sklearn.metrics import classification_report

y_test_true = y_test.tolist()  # nhãn THẬT (chưa bị đổi thành unknown_fallback)

# 1. Các câu bị loại OAN vào "unknown_fallback" (nhãn thật không phải unknown
#    nhưng độ tương đồng dưới threshold nên bị từ chối nhầm)
wrongly_rejected = [
    (X_test_list[i], y_test_true[i], np.max(similarity_matrix[i]))
    for i in range(len(y_test_true))
    if y_pred[i] == "unknown_fallback"
]
print(f"--- Số câu bị THRESHOLD loại oan: {len(wrongly_rejected)} / {len(y_test_true)} "
      f"({len(wrongly_rejected)/len(y_test_true)*100:.1f}%) ---")
if wrongly_rejected:
    print("Ví dụ 5 câu bị loại oan (câu, intent thật, độ tương đồng cao nhất):")
    for ex in sorted(wrongly_rejected, key=lambda x: -x[2])[:5]:
        print(f"  '{ex[0]}' -> thật là [{ex[1]}], score={ex[2]:.3f}")

# 2. Confusion: cặp (nhãn thật -> nhãn dự đoán) bị nhầm nhiều nhất, KHÔNG tính
#    các câu bị loại vào unknown_fallback (để thấy rõ model nhầm lẫn giữa
#    intent nào với intent nào, không lẫn với việc bị threshold chặn)
confusions = [
    (y_test_true[i], y_pred[i])
    for i in range(len(y_test_true))
    if y_pred[i] != y_test_true[i] and y_pred[i] != "unknown_fallback"
]
print(f"\n--- Số câu model nhầm sang intent KHÁC (không tính bị threshold loại): "
      f"{len(confusions)} / {len(y_test_true)} ---")
confusion_counter = Counter(confusions)
print("Top 30 cặp (intent thật -> intent dự đoán) bị nhầm nhiều nhất:")
for (true_label, pred_label), count in confusion_counter.most_common(30):
    print(f"  {true_label}  ->  {pred_label}   ({count} câu)")

# 3. Classification report đầy đủ theo từng intent (precision/recall/f1),
#    dùng y_test_adjusted để khớp đúng với cách tính Accuracy/F1 ở Cell 1
report = classification_report(y_test_adjusted, y_pred, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose().sort_values('f1-score')
print("\n--- 30 intent có F1-score THẤP NHẤT (cần xem lại pattern/dữ liệu) ---")
print(report_df.head(30)[['precision', 'recall', 'f1-score', 'support']])


--- Số câu bị THRESHOLD loại oan: 0 / 1100 (0.0%) ---

--- Số câu model nhầm sang intent KHÁC (không tính bị threshold loại): 118 / 1100 ---
Top 30 cặp (intent thật -> intent dự đoán) bị nhầm nhiều nhất:
  emo_depression  ->  social_symptom   (3 câu)
  social_symptom  ->  emo_depression   (2 câu)
  action_deny  ->  control_switch_topic   (2 câu)
  out_of_scope  ->  crisis_grief_loss   (2 câu)
  faq_4194958  ->  fact-28   (2 câu)
  faq_2746837  ->  faq_1833460   (2 câu)
  faq_5640861  ->  fact-28   (2 câu)
  bot_info  ->  ask_for_help   (1 câu)
  smalltalk_casual  ->  faq_1896541   (1 câu)
  faq_1511075  ->  fact-15   (1 câu)
  emo_depression  ->  where_to_get_help   (1 câu)
  fact-3  ->  fact-32   (1 câu)
  smalltalk_casual  ->  smalltalk_jokes   (1 câu)
  what_is_mental_illness  ->  emo_depression   (1 câu)
  out_of_scope  ->  bot_info   (1 câu)
  mental-health-fact  ->  smalltalk_jokes   (1 câu)
  fact-5  ->  fact-29   (1 câu)
  smalltalk_welcome  ->  action_agree   (1 câu)
  faq_316